# Can the mtanh baseline fit be trusted where the scan acts?

The sparse-grid axes are scale factors on mtanh fit parameters, so a bad fit
does not produce a bad point — it silently redefines what every axis value
means, for the whole scan. This notebook measures the fit where that matters
and nowhere else, then converts the literature bounds into scale factors for
each discharge.

**One parameterization: `fit_mtanh_full`** (Stefanikova 2016, axis-to-SOL in
one formula), driven by `apply_mtanh_full`. The pedestal-only Bruncrona form
was carried alongside it through several passes and lost on every discharge
that separated them — 129038 `ne` 13.3% vs 0.6%, and no case where the
pedestal form won by more than noise. One knob set across every axis and
discharge is worth more to a sparse-grid campaign than a per-axis best form,
so the comparison is retired and `ped` no longer appears here.

**The verdict window is per discharge, and measured.** A single global
`rho_tor` window is the wrong test twice over: `rms_relative` over the whole
profile scores a fit on core structure the scan never touches, and a fixed
0.60–0.95 band asks 132588 (whose $p_e$ is already falling at
$\rho_t\approx0.56$) and 129015 (0.86) the same question. So each discharge
gets its own pedestal region, taken as the quarter-maximum band of
$|\nabla p_e|$ from the data — not from the fit, which is the thing under
test — widened to cover the radii GENE is actually run at. The rms inside that window is what licenses the scale
factors.

Known consequence of standardizing on the full form, and why it is acceptable:
where the fitted SOL floor `b_sol` comes out at zero (Te really does go to ~0
in the SOL), `scale_height` multiplies the whole pedestal component uniformly
and $a/L$ is exactly invariant under multiplication — the height knob moves
the profile but cannot move the drive. Te drive then comes from the WIDTH
axis, which is what Boyle's $\Delta T_e$ bounds are for. Flagged per discharge
in the coverage section.

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import medfilt

from TPED.projects.discharge_tools.src.discharge_data import DischargeData
from TPED.projects.discharge_tools.src.discharge_physics import DischargePhysics
from TPED.projects.discharge_tools.src.transforms.mtanh_transforms import fit_mtanh_full

# Both machines, first one that exists wins — same notebook runs on NERSC and
# on the laptop. Add a path rather than replacing one.
DISCHARGE_ROOT = next(
    (d for d in [r"/global/homes/j/joeschm/data/ST_research/NSTXU_discharges",
                 r"C:/Users/joesc/git/ST_research/NSTXU_discharges"]
     if os.path.isdir(d)), None)
if DISCHARGE_ROOT is None:
    raise FileNotFoundError("no discharge root found; add this machine's path")

# 129038's directory holds five pfiles, so auto-discovery refuses to guess.
DISCHARGES = {129015: {}, 129038: {"pfile": "p129038.00400"},
              132543: {}, 132588: {}}

# Profiles the axes act on, plus the ones that follow through quasineutrality —
# a fit fine for ne and wrong for ni still reaches CHEASE.
VARS = ["Te", "Ti", "ne", "ni"]

FIT_KWARGS = dict(pedestal_weight=8.0)   # matches the campaign
TRUST = 0.01                             # rms <= 1% of profile range, in-window

# Where GENE is actually run. 132543/132588's radii are the q=4 and q=5
# surfaces and sit inside the pedestal top, so the verdict window has to reach
# them even when the pedestal proper starts further out.
ANALYSIS_RADII = {129015: (0.85,), 129038: (0.85,),
                  132543: (0.736, 0.825), 132588: (0.736, 0.825)}

SCALE_BOX = (0.7, 1.3)     # campaign scale box, for the coverage section

# Pedestal-region detection, on |grad pe| from the DATA (the fit is under test).
# A quarter-maximum band rather than half: half-max cut 132588 off at 0.625 when
# its pe profile is visibly falling from ~0.55, because a single-point spike at
# 0.740 sets the maximum the threshold is taken against. The median filter kills
# that spike; the looser fraction makes the window conservative, which is the
# right direction for a test the fit has to pass.
GRAD_FRAC   = 0.25         # quarter-maximum band around the peak gradient
GRAD_MEDFILT = 5           # points; spike rejection before thresholding
GRAD_SEARCH = (0.50, 0.999)   # 132543's core gradient dominates below this
WINDOW_CAP  = 0.99         # never score past here: pfile SOL is untrustworthy
WINDOW_MIN_HI = 0.95       # always score at least out to here

# fit_mtanh_full bounds b_pos to [0.85, 0.999]. A fit landing exactly on 0.85
# has hit the bound, not found the pedestal — reported, not silently used.
BPOS_BOUND = 0.85

## Fit, and locate each discharge's pedestal

One pass: fit `Te`, `Ti`, `ne`, `ni` and the derived `pe = ne*Te` with
`fit_mtanh_full`, then set the windows.

- **pedestal region** — quarter-maximum band of $|\nabla p_e|$, computed with
  TPED's `gradient_length` (4th-order non-uniform) on the measured profile and
  median-filtered before thresholding. Data-driven on purpose: deriving it from
  `b_pos ± 2 b_width` would let a bad fit choose the window that judges it, and
  three of these four discharges return `b_pos` pinned on the fitter's 0.85
  bound. Quarter rather than half-maximum because a half-max cut on the raw
  gradient put 132588's inner edge at 0.625 while its $p_e$ is visibly falling
  from ~0.55 — one spiky point at 0.740 was setting the threshold the rest of
  the band was measured against. The looser band errs wide, which is the safe
  direction for a test the fit has to pass.
- **verdict window** — the pedestal region widened to cover `ANALYSIS_RADII`
  and out to at least 0.95, capped at 0.99.

`Ti` and `ni` are fitted, plotted and scored alongside the axis variables: no
scan axis names them, but quasineutrality and the closure carry them into
CHEASE regardless.

Errors are normalized by profile RANGE, not by $y$: a relative error against a
value going to ~0 at the separatrix explodes there and swamps the region under
test.

In [ ]:
def load(shot):
    """DischargePhysics for one shot, with pe = ne*Te added as a variable so
    TPED's fitter and gradient_length see it like any other profile."""
    d = os.path.join(DISCHARGE_ROOT, str(shot))
    kw = {"input_dir": d}
    if DISCHARGES[shot].get("pfile"):
        kw["pfile"] = os.path.join(d, DISCHARGES[shot]["pfile"])
    phys = DischargePhysics(DischargeData(**kw))
    ds = phys.ds.copy()
    ds["pe"] = phys.ds["ne"] * phys.ds["Te"]
    return DischargePhysics(ds)


def ped_region(phys, var="pe", frac=GRAD_FRAC, search=GRAD_SEARCH):
    """Quarter-maximum band of |grad var| — the pedestal, from the data.

    Returns (lo, peak, hi) in rho_tor. Median-filtered first so one noisy point
    cannot set the maximum the threshold is measured against, and contiguous
    around the peak, so a second steep feature further in does not silently
    annex the window.
    """
    x = np.asarray(phys.rhot.values)
    y = np.asarray(phys.ds[var].values, dtype=float)
    g = medfilt(np.abs(y / np.asarray(phys.gradient_length(var).values)),
                GRAD_MEDFILT)
    m = (x >= search[0]) & (x <= search[1])
    xs, gs = x[m], g[m]
    i = int(np.nanargmax(gs))
    lo = i
    while lo > 0 and gs[lo - 1] >= frac * gs[i]:
        lo -= 1
    hi = i
    while hi < len(gs) - 1 and gs[hi + 1] >= frac * gs[i]:
        hi += 1
    return float(xs[lo]), float(xs[i]), float(xs[hi])


def fit_record(phys, var, rec, **kw):
    """Fit one profile and return the fields the rest of the notebook reads."""
    x = np.asarray(phys.rhot.values)
    try:
        profile, meta = fit_mtanh_full(phys.ds, var, **kw)
    except Exception as exc:
        return {"error": f"{type(exc).__name__}: {exc}"}
    yhat = np.asarray(profile(x), dtype=float)
    fp = meta["fit_params"]
    return {"yhat": yhat, "err": (yhat - rec["y"]) / rec["scale"],
            "rms_global": meta["rms_relative"], "params": fp,
            "pinned": abs(fp["b_pos"] - BPOS_BOUND) < 1e-6}


def window_stats(x, err, win):
    """rms and max |error| inside a window, as a fraction of profile range."""
    e = err[(x >= win[0]) & (x <= win[1])]
    return float(np.sqrt(np.mean(e ** 2))), float(np.max(np.abs(e)))


results = {}
for shot in DISCHARGES:
    phys = load(shot)
    x = np.asarray(phys.rhot.values)
    lo, pk, hi = ped_region(phys)
    win = (min([lo] + list(ANALYSIS_RADII[shot])),
           min(max(hi, WINDOW_MIN_HI), WINDOW_CAP))
    entry = {"phys": phys, "x": x, "ped": (lo, pk, hi), "win": win, "vars": {}}
    for var in VARS + ["pe"]:
        y = np.asarray(phys.ds[var].values, dtype=float)
        rec = {"y": y, "scale": float(np.max(y) - np.min(y)) or 1.0,
               "kwargs": dict(FIT_KWARGS)}
        rec.update(fit_record(phys, var, rec, **FIT_KWARGS))
        entry["vars"][var] = rec
        if "err" not in rec:
            print(f"  {shot} {var}: FIT FAILED — {rec['error']}")
    results[shot] = entry

shots = sorted(results)
varlist = [v for v in VARS + ["pe"] if any(v in e["vars"] for e in results.values())]

print(f"{'shot':>7}  {'pedestal region':>17} {'peak':>6}  {'verdict window':>15}"
      f"  {'GENE radii':>13}")
print("-" * 70)
for s in shots:
    lo, pk, hi = results[s]["ped"]
    w = results[s]["win"]
    radii = ",".join(f"{r:.3f}" for r in ANALYSIS_RADII[s])
    print(f"{s:>7}  {lo:>8.3f}-{hi:<8.3f} {pk:>6.3f}  {w[0]:>7.3f}-{w[1]:<7.3f}"
          f"  {radii:>13}")
print("\npedestal region = quarter-max band of |grad pe| from the data; "
      "verdict window = that band widened to the GENE radii and to 0.95, "
      f"capped at {WINDOW_CAP}")

## The verdict — error inside each discharge's own window

`rms_win` / `max_win` are taken over the verdict window; `rms_ped` over the
pedestal region alone; `rms_in` over the stretch immediately inboard of the
window (0.50 to its inner edge); `rms_global` is the whole-profile number the
first pass used, kept for contrast. Where `rms_global` and `rms_win` diverge,
the global number was measuring something the scan does not touch.

`rms_in` is not part of the verdict but is the honest check on it: a profile
that passes in-window while carrying several percent just inboard — 129015
`Ti` is the case here — is being flattered by a narrow window, and for
`Ti`/`ni` that error still reaches CHEASE through the closure.

`b_pos` is printed because a value sitting exactly on 0.85 is the fitter's
lower bound, not a located pedestal — that fit can still score well and still
be the wrong shape to scale, and it is the failure mode behind the runaway
`ne_width_scale` numbers further down.

In [ ]:
rows = []
for shot in shots:
    e = results[shot]
    x, win, ped = e["x"], e["win"], e["ped"][0::2]
    for var in varlist:
        f = e["vars"].get(var, {})
        if "err" not in f:
            rows.append({"shot": shot, "var": var, "rms_win": np.nan,
                         "max_win": np.nan, "rms_ped": np.nan, "rms_in": np.nan,
                         "rms_global": np.nan, "b_pos": np.nan,
                         "pinned": False, "ok": False,
                         "note": f.get("error", "")[:40]})
            continue
        rms, mx = window_stats(x, f["err"], win)
        # Error immediately INBOARD of the window, GRAD_SEARCH[0] to the window
        # edge. Not part of the verdict -- no axis acts there -- but a large
        # number here means the fit is only good because the window is narrow,
        # and for Ti/ni it still reaches CHEASE through the closure.
        inb = (window_stats(x, f["err"], (GRAD_SEARCH[0], win[0]))[0]
               if win[0] > GRAD_SEARCH[0] else np.nan)
        rows.append({"shot": shot, "var": var, "rms_win": rms, "max_win": mx,
                     "rms_ped": window_stats(x, f["err"], ped)[0], "rms_in": inb,
                     "rms_global": f["rms_global"], "b_pos": f["params"]["b_pos"],
                     "pinned": f["pinned"], "ok": rms <= TRUST, "note": ""})

hdr = (f"{'shot':>7} {'var':<4} {'rms_win':>8} {'max_win':>8} {'rms_ped':>8} "
       f"{'rms_in':>7} {'rms_global':>10} {'b_pos':>7} {'verdict':>8}")
print(hdr); print("-" * len(hdr))
for r in rows:
    nan = lambda v, w: (f"{v*100:>{w-1}.2f}%" if v == v else f"{'--':>{w}}")
    print(f"{r['shot']:>7} {r['var']:<4} {nan(r['rms_win'],8)} {nan(r['max_win'],8)} "
          f"{nan(r['rms_ped'],8)} {nan(r['rms_in'],7)} {nan(r['rms_global'],10)} "
          f"{r['b_pos']:>7.4f} {'TRUST' if r['ok'] else 'reject':>8}"
          + ("   b_pos ON BOUND" if r["pinned"] else "")
          + (f"   {r['note']}" if r["note"] else ""))

print(f"\nTRUST: rms inside the discharge's verdict window <= {TRUST:.0%} of range")
print(f"{'shot':>7}  {'Te':>7} {'ne':>7} {'pe':>7}   axes usable?")
print("-" * 48)
verdict = {}
for shot in shots:
    v = {var: next((r["rms_win"] for r in rows
                    if r["shot"] == shot and r["var"] == var), np.nan)
         for var in ("Te", "ne", "pe")}
    ok = all(t == t and t <= TRUST for t in v.values())
    verdict[shot] = {**{f"{k}_rms_win": t for k, t in v.items()},
                     "window": list(results[shot]["win"]),
                     "ped_region": list(results[shot]["ped"][0::2]), "ok": ok}
    print(f"{shot:>7}  " + " ".join(f"{t*100:>6.2f}%" for t in v.values())
          + f"   {'yes' if ok else 'NO — refit before scaling'}")

with open("mtanh_fit_quality.json", "w") as fh:
    json.dump({"form": "full", "trust_threshold": TRUST,
               "rows": rows, "verdict": verdict}, fh, indent=1, default=str)
print("\nwritten: mtanh_fit_quality.json")

## Look at it

Left block: fits against data over each discharge's own pedestal region
(shaded = verdict window, dotted = GENE radii). Right block: residual against
radius, core to SOL, so a fit that is poor in the core and sound in the window
is visibly that rather than assumed to be. Bottom: the verdict, one bar per
profile, threshold drawn.

What disqualifies a fit is error *rising inside the shaded band*. Error in the
core is not a defect here — no scan axis acts there.

In [ ]:
COL = {"Te": "tab:red", "Ti": "tab:orange", "ne": "tab:blue",
       "ni": "tab:cyan", "pe": "tab:purple"}
PLOT_VARS = ["Te", "Ti", "ne", "ni", "pe"]   # Ti/ni included: they reach
#   CHEASE through quasineutrality and the closure even though no axis names them

fig, axes = plt.subplots(len(PLOT_VARS) + 1, len(shots),
                         figsize=(3.5 * len(shots), 2.2 * (len(PLOT_VARS) + 1)),
                         squeeze=False)
for j, shot in enumerate(shots):
    e = results[shot]
    x, win = e["x"], e["win"]
    lo, pk, hi = e["ped"]
    for i, var in enumerate(PLOT_VARS):
        ax = axes[i][j]
        f = e["vars"].get(var, {})
        m = (x >= win[0] - 0.15) & (x <= 1.0)
        ax.plot(x[m], f["y"][m], ".", ms=3, color="0.55", label="data")
        if "yhat" in f:
            ax.plot(x[m], f["yhat"][m], "-", lw=1.5, color=COL[var], label="full fit")
            r = window_stats(x, f["err"], win)[0]
            ax.text(0.03, 0.08, f"rms {r*100:.2f}%", fontsize=8,
                    transform=ax.transAxes,
                    color="green" if r <= TRUST else "tab:red")
        ax.axvspan(*win, color="tab:green", alpha=0.10)
        ax.axvline(pk, color="k", lw=0.7, ls="-.", alpha=0.5)
        for r0 in ANALYSIS_RADII[shot]:
            ax.axvline(r0, color="k", lw=0.8, ls=":", alpha=0.8)
        ax.set_xlim(win[0] - 0.15, 1.0)
        ax.tick_params(labelsize=7)
        if i == 0:
            ax.set_title(f"{shot}   ped {lo:.2f}-{hi:.2f}", fontsize=10)
        if j == 0:
            ax.set_ylabel(var)
    # residual row, full radius, every variable
    ax = axes[-1][j]
    for var in varlist:
        f = e["vars"].get(var, {})
        if "err" in f:
            ax.plot(x, 100 * f["err"], lw=1.1, color=COL[var], label=var, alpha=0.9)
    ax.axvspan(*win, color="tab:green", alpha=0.10)
    ax.axhline(0, color="k", lw=0.6)
    ax.set_xlim(0, 1.0); ax.set_ylim(-8, 8)
    ax.set_xlabel("rho_tor"); ax.tick_params(labelsize=7)
    if j == 0:
        ax.set_ylabel("(fit - data)/range [%]")
axes[0][-1].legend(fontsize=6)
axes[-1][-1].legend(fontsize=6, ncol=2)
fig.suptitle("Stefanikova full-form fits — shaded: verdict window, "
             "dotted: GENE radii, dash-dot: peak |grad pe|")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(10, 3.4))
labels = [f"{r['shot']} {r['var']}" for r in rows]
ax.bar(np.arange(len(rows)), [100 * r["rms_win"] for r in rows], 0.6,
       color=[COL[r["var"]] for r in rows])
ax.axhline(100 * TRUST, color="k", ls="--", lw=1.2, label=f"trust {TRUST:.0%}")
ax.set_xticks(np.arange(len(rows)))
ax.set_xticklabels(labels, fontsize=6, rotation=90)
ax.set_yscale("log"); ax.set_ylabel("rms in verdict window [% of range]")
ax.legend(fontsize=8)
ax.set_title("per-discharge verdict — below the dashed line is usable")
plt.tight_layout(); plt.show()

### If a profile is rejected

In order of effort; re-run the cells above after each, the verdict table is the
arbiter.

1. **Raise `pedestal_weight`** (currently 8) — upweights points inside
   `ped_threshold`–`edge_threshold`, buying pedestal accuracy at the core's
   expense, which is the trade this window says we want.
2. **Lower `ped_threshold`** (default 0.85) — for the discharges whose `b_pos`
   is pinned on the 0.85 bound, the pedestal genuinely sits further in and the
   fitter is not allowed to look there. This is the first thing to try for
   129038 and 132588.
3. **Supply explicit `p0`** — eight parameters `[b_height, b_sol, b_pos,
   b_width, b_slope, a_height, a_width, a_exp]`. Seed `b_pos` and `b_width`
   from the measured peak-gradient location printed above; those two are what
   the auto-guess gets wrong.

**Tried and rejected (2026-08-22): per-profile tuning of `ped_threshold`,
seeded `p0`, and `b_pos` bounds freed to the measured region, swept over
`pedestal_weight` 8–64.** It worked on the metric this notebook reports — every
profile passed in-window, worst rms 0.74%, 129038 `pe` went 4.54% to 0.21% —
and it was reverted anyway. Freeing `b_pos` inward (0.68–0.76 on 129038 and
132588) buys pedestal accuracy by letting the mtanh reinterpret half the
profile as pedestal, and the resulting reconstruction is distorted outside the
pedestal. **CHEASE-BS consumes the whole reconstructed profile, not the window
this notebook scores**, so a fit that wins in-window and wanders in the core is
worse for the campaign than one that is mediocre in-window and stable
everywhere. The single campaign setting — `pedestal_weight=8`, default
`ped_threshold`, default bounds — is what is used, and rejections are addressed
by steps 1–3 one profile at a time rather than by an automated sweep.

## What span do the axes actually buy? — $a/L_{p_e}$ at the analysis radius

Everything below is the scaling half, unchanged in intent and now single-form.
The only quantity that reaches the physics is what GENE computes from the
profiles it is handed, and for drive that is the normalized inverse scale
length — measured with TPED's `gradient_length`, not a local difference.

A flat line is an axis that cannot move the drive at that radius: the scan is
a no-op there whatever the fit quality. Non-monotonicity is a separate problem
— the sparse grid assumes the QoI is smooth in the axes.

In [ ]:
def a_over_L_pe(phys, x0):
    """a/L for pe at one radius, from TPED's 4th-order non-uniform derivative."""
    ds = phys.ds.copy()
    ds["pe"] = phys.ds["ne"] * phys.ds["Te"]
    p = DischargePhysics(ds)
    x = np.asarray(p.rhot.values)
    L = np.asarray(p.gradient_length("pe").values)
    return float(1.0 / L[int(np.argmin(np.abs(x - x0)))])


scales = np.linspace(SCALE_BOX[0], SCALE_BOX[1], 7)
curves, span_rows = {}, []
for shot in shots:
    phys = results[shot]["phys"]
    fits = {v: fit_mtanh_full(phys.ds, v, **FIT_KWARGS)[0] for v in ("Te", "ne")}
    for x0 in ANALYSIS_RADII[shot]:
        for axis in ("Te", "ne"):
            vals = [a_over_L_pe(
                phys.apply_mtanh_full(axis, fit=fits[axis], scale_height=s,
                                      enforce_quasineutrality=True, qz=6.0), x0)
                for s in scales]
            curves[(shot, x0, axis)] = vals
            base = vals[len(scales) // 2]
            span_rows.append({"shot": shot, "x0": x0, "axis": axis,
                              "span": max(vals) - min(vals),
                              "span_pct": 100 * (max(vals) - min(vals)) / abs(base)})

print(f"a/L_pe span across scale_height {SCALE_BOX[0]}-{SCALE_BOX[1]}")
print(f"{'shot':>7} {'x0':>6} {'axis':<4} {'span':>8} {'span %':>8}  {'b_sol':>11}")
print("-" * 52)
for r in span_rows:
    bs = results[r["shot"]]["vars"][r["axis"]]["params"]["b_sol"]
    flag = "  <-- b_sol=0: height axis cannot move a/L" if abs(bs) < 1e-9 else ""
    print(f"{r['shot']:>7} {r['x0']:>6.3f} {r['axis']:<4} {r['span']:>8.3f} "
          f"{r['span_pct']:>7.1f}% {bs:>11.4g}{flag}")

pairs = [(s, x0) for s in shots for x0 in ANALYSIS_RADII[s]]
fig, axes = plt.subplots(1, len(pairs), figsize=(2.6 * len(pairs), 3.2),
                         squeeze=False)
for ax, (shot, x0) in zip(axes[0], pairs):
    for axis in ("Te", "ne"):
        ax.plot(scales, curves[(shot, x0, axis)], "-", color=COL[axis],
                lw=1.3, label=axis)
    ax.set_title(f"{shot}  x0={x0}", fontsize=9)
    ax.set_xlabel("scale_height"); ax.tick_params(labelsize=7)
axes[0][0].set_ylabel("a/L_pe")
axes[0][-1].legend(fontsize=6)
fig.suptitle("KBM-relevant drive reachable by each height axis — flat means unreachable")
plt.tight_layout(); plt.show()

## Absolute-unit bounds — Boyle 2011, converted per discharge

The handover: take the survey bounds in the units they are quoted in, convert
them to the scale factors this discharge's fit needs, and check the target is
reachable before a campaign spends CHEASE time discovering it is not. Scale
factors are an implementation detail of the transform — they differ per
discharge and per variable, and quoting a scan box in them is how the last
three rounds of confusion started.

Widths are Boyle 2011 PPCF (lithium scan, Fig. 7), which separates two regimes
— different plasmas, not one range, so they are scanned as separate boxes.

The metric per axis is the physical quantity the bound is quoted in: pedestal
-top value for heights, full width in %$\psi_N$ for widths, pedestal-top ratio
for `Ti_Te`. The scale→metric map is linear to machine precision, so one probe
point inverts it exactly: `scale = 1 + (target/nominal - 1)/slope`.

In [ ]:
# Boyle 2011 PPCF, Fig. 7: pedestal FULL widths in % psi_N.
#   7a = Delta_ne, 7d = Delta_Te, 7g = Delta_pe
BOYLE_WIDTHS = {
    "ELMy":     {"dne": (6, 12),  "dTe": (4, 7),   "dpe": (4, 8)},
    "ELM_free": {"dne": (14, 22), "dTe": (7, 10),  "dpe": (8, 12)},
}

# Pedestal-top values and the ion/electron ratio; frozen as one box because
# together with the widths they set beta.
TARGETS = {"Te_ped": (0.2, 0.8),   # keV at B0 ~ 0.4 T
           "ne_ped": (3.0, 7.0),   # 1e19 m^-3
           "Ti_Te":  (1.0, 2.0)}

# "auto" classifies each discharge from its own nominal widths: the regime is a
# property of the plasma. Scored against ELMy, 129038 -- ne 18.6%, pe 9.0%,
# squarely ELM-free -- produced width scale factors of 0.20-0.37, an
# instruction to shrink an ELM-free pedestal to a fifth of its width.
REGIME = "auto"          # "auto", "ELMy", "ELM_free", or {shot: regime}
SCALE_SANITY = (0.3, 3.0)   # beyond this nothing has been tested against cheaseBS

# axis -> (variable, transform kwarg, metric, unit, display factor)
AXES = {
    "Te_ped_scale":   ("Te", "scale_height", "ped_top", "keV",   1e-3),
    "ne_ped_scale":   ("ne", "scale_height", "ped_top", "1e19",  1e-19),
    "Te_width_scale": ("Te", "scale_width",  "width",   "%psiN", 1.0),
    "ne_width_scale": ("ne", "scale_width",  "width",   "%psiN", 1.0),
    "Ti_Te_scale":    ("Ti", "scale_height", "ti_te",   "-",     1.0),
}


def _apply(phys, var, fit, kwarg, s):
    return phys.apply_mtanh_full(var, fit=fit, **{kwarg: s},
                                 enforce_quasineutrality=True, qz=6.0)


def _ped_top(phys, var, shot):
    """Profile value at this discharge's own pedestal top (inner edge of its
    measured pedestal region) — not a fixed 0.90-0.98 window."""
    x = np.asarray(phys.rhot.values)
    y = np.asarray(phys.ds[var].values, dtype=float)
    return float(y[int(np.argmin(np.abs(x - results[shot]["ped"][0])))])


def _width_psin(phys, var):
    """Full pedestal width in % psi_N.

    Boyle quotes widths in poloidal flux, the fit works in rho_tor, and the
    conversion factor is not a constant — it depends where the pedestal sits.
    Stefanikova's b_width is a QUARTER width (full = 4*b_width) about b_pos.
    """
    ds = phys.ds.copy()
    if var == "pe":
        ds["pe"] = phys.ds["ne"] * phys.ds["Te"]
    fp = fit_mtanh_full(ds, var, **FIT_KWARGS)[1]["fit_params"]
    rhot = np.asarray(phys.rhot.values)
    rhop = np.asarray(phys.rhop.values)
    w = 4.0 * fp["b_width"]
    pl, ph = np.interp([fp["b_pos"] - w / 2, fp["b_pos"] + w / 2], rhot, rhop)
    return 100.0 * (ph ** 2 - pl ** 2)


def metric_of(phys, var, kind, shot):
    if kind == "ped_top":
        return _ped_top(phys, var, shot)
    if kind == "width":
        return _width_psin(phys, var)
    if kind == "ti_te":
        return _ped_top(phys, "Ti", shot) / _ped_top(phys, "Te", shot)
    raise ValueError(kind)


def axis_response(phys, axis, fits, shot, probe=1.2):
    """Nominal metric (in the units the bounds are quoted in) and its linear
    slope in the scale factor. The slope is relative, so unit conversion does
    not touch it; the nominal must be converted or the inversion is meaningless."""
    var, kwarg, kind, unit, disp = AXES[axis]
    at = lambda s: metric_of(_apply(phys, var, fits[var], kwarg, s), var, kind, shot)
    m0 = at(1.0)
    return m0 * disp, (at(probe) - m0) / m0 / (probe - 1.0), unit, disp


def scale_for(m0, slope, target):
    """Invert the linear map; nan when the axis cannot move the metric."""
    if not np.isfinite(slope) or abs(slope) < 1e-6:
        return float("nan")
    return 1.0 + ((target / m0) - 1.0) / slope


def classify_regime(phys):
    """Boyle band the nominal widths sit in. Votes across dTe, dne, dpe."""
    widths = {"dTe": _width_psin(phys, "Te"), "dne": _width_psin(phys, "ne"),
              "dpe": _width_psin(phys, "pe")}
    scores = {}
    for name, bands in BOYLE_WIDTHS.items():
        inside = sum(bands[k][0] <= w <= bands[k][1] for k, w in widths.items())
        dist = sum(0.0 if bands[k][0] <= w <= bands[k][1]
                   else min(abs(w - bands[k][0]), abs(w - bands[k][1]))
                   for k, w in widths.items())
        scores[name] = (inside, -dist)
    return max(scores, key=scores.get), widths, scores


print("axes:", ", ".join(AXES))

### Nominal values against the target box

`nominal` is where each discharge already sits; `need lo` / `need hi` are the
scale factors that would reach the bound. A nominal inside its target range is
the comfortable case; outside means the axis has to travel one way only.

In [ ]:
SHOT_REGIME, FITS, WIDTHS = {}, {}, {}
print("regime classification:")
for shot in shots:
    phys = results[shot]["phys"]
    FITS[shot] = {v: fit_mtanh_full(phys.ds, v, **FIT_KWARGS)[0]
                  for v in ("Te", "Ti", "ne")}
    if isinstance(REGIME, dict):
        SHOT_REGIME[shot] = REGIME[shot]
        continue
    if REGIME != "auto":
        SHOT_REGIME[shot] = REGIME
        continue
    best, widths, scores = classify_regime(phys)
    SHOT_REGIME[shot], WIDTHS[shot] = best, widths
    print(f"  {shot}: {best:<9} ("
          + ", ".join(f"{k} {v:.1f}%" for k, v in widths.items()) + ";  "
          + ", ".join(f"{n} {s[0]}/3 in" for n, s in scores.items()) + ")")

bounds_per_shot, reach_rows = {}, []
for shot in shots:
    phys = results[shot]["phys"]
    per_axis = {}
    for axis in AXES:
        var, kwarg, kind, unit, disp = AXES[axis]
        if kind == "width":
            lo, hi = BOYLE_WIDTHS[SHOT_REGIME[shot]]["dTe" if var == "Te" else "dne"]
        else:
            lo, hi = TARGETS["Ti_Te" if kind == "ti_te" else f"{var}_ped"]
        m0, slope, unit, disp = axis_response(phys, axis, FITS[shot], shot)
        s_lo, s_hi = sorted((scale_for(m0, slope, lo), scale_for(m0, slope, hi)))
        per_axis[axis] = {"nominal": m0, "unit": unit, "slope": slope,
                          "target": (lo, hi), "scale": (s_lo, s_hi),
                          "ok": bool(np.isfinite(s_lo) and np.isfinite(s_hi)
                                     and SCALE_SANITY[0] <= s_lo <= SCALE_SANITY[1]
                                     and SCALE_SANITY[0] <= s_hi <= SCALE_SANITY[1])}
        reach_rows.append({"shot": shot, "axis": axis, **per_axis[axis]})
    bounds_per_shot[shot] = per_axis

hdr = (f"{'shot':>7} {'axis':<16} {'nominal':>9} {'unit':<6} {'target':>11} "
       f"{'slope':>7} {'need lo':>8} {'need hi':>8}  reach")
print(); print(hdr); print("-" * len(hdr))
for r in reach_rows:
    lo, hi = r["target"]; sl, sh = r["scale"]
    print(f"{r['shot']:>7} {r['axis']:<16} {r['nominal']:>9.3f} {r['unit']:<6} "
          f"{f'{lo}-{hi}':>11} {r['slope']:>7.3f} {sl:>8.2f} {sh:>8.2f}  "
          f"{'ok' if r['ok'] else 'OUT OF REACH'}")
print(f"\nreachable = both scale factors inside {SCALE_SANITY}")

### The scan box, in scale factors, ready for ScanStudy

Edges are trimmed by physics rather than clipped at a round number: an edge is
walked back toward nominal until the profile it produces is one Boyle could
have observed — positive, monotonic outward through the pedestal, and with a
$p_e$ width in band. `pe` gets no axis (it is derived, and CHEASE builds the
pressure from the profiles regardless) but Boyle's panel 7g makes it a free
consistency filter on the corners.

In [ ]:
def edge_ok(phys0, fits, axis, s, shot):
    """Is this box edge a plasma worth submitting?

    Physical tests, not an arbitrary scale ceiling: a scale factor is out of
    bounds when the profile it produces stops being one Boyle observed.
    """
    var, kwarg, kind, _, _ = AXES[axis]
    try:
        q = _apply(phys0, var, fits[var], kwarg, s)
        y = np.asarray(q.ds[var].values, dtype=float)
        if float(np.min(y)) <= 0:
            return False                       # went non-positive
        # Monotonic decrease outward through the pedestal. A large height
        # scaling can push the Stefanikova core Gaussian and the pedestal
        # plateau out of proportion and raise a bump at rho~0.9 that the width
        # filter cannot see -- a bump changes shape without changing width.
        x = np.asarray(q.rhot.values)
        m = (x >= 0.6) & (x <= 1.0)
        if np.any(np.diff(y[m]) > 0.02 * float(np.max(y[m]))):
            return False
        w = _width_psin(q, "pe")
    except Exception:
        return False
    lo_pe, hi_pe = BOYLE_WIDTHS[SHOT_REGIME[shot]]["dpe"]
    if lo_pe <= w <= hi_pe:
        return True
    # Some nominals start outside Boyle's band. Demanding the band outright
    # would empty their boxes and say nothing; the test becomes "do not make it
    # worse", leaving out-of-band-at-nominal as the separate finding it is.
    nom = WIDTHS.get(shot, {}).get("dpe")
    if nom is None or lo_pe <= nom <= hi_pe:
        return False
    dev = min(abs(w - lo_pe), abs(w - hi_pe))
    return dev <= min(abs(nom - lo_pe), abs(nom - hi_pe)) * 1.25


def trim_edge(phys0, fits, axis, s_target, shot, s_from=1.0, tol=0.01):
    """Bisect an edge back toward nominal until it passes edge_ok."""
    if edge_ok(phys0, fits, axis, s_target, shot):
        return s_target
    lo, hi = s_from, s_target
    if not edge_ok(phys0, fits, axis, lo, shot):
        return None
    while abs(hi - lo) > tol:
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if edge_ok(phys0, fits, axis, mid, shot) else (lo, mid)
    return round(lo, 4)


SG_BOUNDS = {}
for shot in shots:
    phys0, fits = results[shot]["phys"], FITS[shot]
    WIDTHS.setdefault(shot, {}).setdefault("dpe", _width_psin(phys0, "pe"))
    box, notes = {}, []
    for axis, r in bounds_per_shot[shot].items():
        sl, sh = r["scale"]
        if not (np.isfinite(sl) and np.isfinite(sh)):
            notes.append(f"{axis}: axis cannot move its metric — dropped")
            continue
        # Clip to the tested scale range first: nothing outside it has been run
        # against cheaseBS, so trimming from an untested edge would bisect
        # through profiles we have no basis to judge.
        sl, sh = max(sl, SCALE_SANITY[0]), min(sh, SCALE_SANITY[1])
        if sh <= sl:
            notes.append(f"{axis}: target lies outside the tested scale range "
                         f"{SCALE_SANITY} — dropped")
            continue
        cl = trim_edge(phys0, fits, axis, sl, shot)
        ch = trim_edge(phys0, fits, axis, sh, shot)
        if cl is None or ch is None or ch <= cl:
            notes.append(f"{axis}: no usable range — even small moves off "
                         "nominal leave Boyle's pe band")
            continue
        if (cl, ch) != (sl, sh):
            lo, hi = r["target"]
            notes.append(
                f"{axis}: trimmed to physics, reaches "
                f"{r['nominal'] * (1 + r['slope'] * (cl - 1)):.3g}-"
                f"{r['nominal'] * (1 + r['slope'] * (ch - 1)):.3g} {r['unit']} "
                f"of {lo}-{hi}")
        box[axis] = (round(cl, 4), round(ch, 4))
    SG_BOUNDS[shot] = box
    print(f"{shot}: {json.dumps(box)}")
    for n in notes:
        print(f"    ! {n}")

with open("sg_bounds.json", "w") as fh:
    json.dump({"form": "full",
               "regime": {str(k): v for k, v in SHOT_REGIME.items()},
               "targets": TARGETS, "boyle_widths": BOYLE_WIDTHS,
               "scale_sanity": list(SCALE_SANITY),
               "bounds": {str(k): v for k, v in SG_BOUNDS.items()}}, fh, indent=1)
print("\nwritten: sg_bounds.json")

### Corners: pressure-width check and a look before handing over

A corner whose scanned `ne` and `Te` widths imply a $p_e$ width outside Boyle's
band is not a plasma he observed — a box edge to pull in, not a run to submit.
The curves should separate visibly, stay physical, and span what the table
claims: a corner overlapping nominal is an axis contributing nothing.

In [ ]:
print(f"{'shot':>7} {'corner':<26} {'pe width':>9}  band  verdict")
print("-" * 62)
pe_rows = []
for shot in shots:
    phys0, fits = results[shot]["phys"], FITS[shot]
    lo_pe, hi_pe = BOYLE_WIDTHS[SHOT_REGIME[shot]]["dpe"]
    corners = [("nominal", {})]
    for axis, (blo, bhi) in SG_BOUNDS[shot].items():
        if AXES[axis][0] in ("Te", "ne"):
            corners += [(f"{axis} lo ({blo:.2f})", {axis: blo}),
                        (f"{axis} hi ({bhi:.2f})", {axis: bhi})]
    for label, point in corners:
        phys = phys0
        for axis, s in point.items():
            var, kwarg, _, _, _ = AXES[axis]
            phys = _apply(phys, var, fits[var], kwarg, s)
        try:
            w = _width_psin(phys, "pe")
        except Exception as exc:
            print(f"{shot:>7} {label:<26} {'--':>9}  fit failed: {type(exc).__name__}")
            continue
        ok = lo_pe <= w <= hi_pe
        pe_rows.append({"shot": shot, "corner": label, "pe_width": w, "ok": ok})
        print(f"{shot:>7} {label:<26} {w:>8.1f}%  {lo_pe}-{hi_pe}  "
              f"{'ok' if ok else 'OUTSIDE Boyle'}")
bad = sum(not r["ok"] for r in pe_rows)
print(f"\n{bad}/{len(pe_rows)} corners outside their discharge's pe width band")

fig, axg = plt.subplots(2, len(shots), figsize=(3.5 * len(shots), 5.6),
                        squeeze=False)
for j, shot in enumerate(shots):
    phys, fits = results[shot]["phys"], FITS[shot]
    x = np.asarray(phys.rhot.values)
    for row, var in enumerate(("Te", "ne")):
        ax = axg[row][j]
        ax.plot(x, np.asarray(phys.ds[var].values, dtype=float), "-", lw=2.0,
                color="k", label="nominal", zorder=3)
        for axis, (lo, hi) in SG_BOUNDS[shot].items():
            if AXES[axis][0] != var:
                continue
            kwarg = AXES[axis][1]
            for s, ls in ((lo, "--"), (hi, ":")):
                q = _apply(phys, var, fits[var], kwarg, s)
                ax.plot(x, np.asarray(q.ds[var].values, dtype=float), ls, lw=1.2,
                        label=f"{kwarg[6:]} {s:.2f}")
        ax.axvspan(*results[shot]["win"], color="tab:green", alpha=0.08)
        ax.set_xlim(0.6, 1.0); ax.set_ylabel(var)
        ax.tick_params(labelsize=7); ax.legend(fontsize=6)
        if row == 0:
            ax.set_title(str(shot), fontsize=10)
        else:
            ax.set_xlabel("rho_tor")
fig.suptitle("scan box corners against nominal — verify before handing to the sparse grid")
plt.tight_layout(); plt.show()